# Mapa

Mapa ČR zobrazující obce/volební okrsky a jejich podíl prefenčních hlasů pro ženy. 
Na úrovni volebních okrsků jsou výsledky vidět i pro jednotlivé městské části, pokud je města mají. 

In [ ]:
import geopandas as gpd

okrsky = gpd.read_file("data_vstup/vol_okrsky_2025_20250701.geojson")
hranice_cr = okrsky.dissolve()
hranice_cr.to_file("hranice_cr.geojson", driver="GeoJSON")
print("Hotovo!")

Hotovo!


In [25]:
# mapa obce
import folium
import pandas as pd

obce = pd.read_csv("data_vystup/centroidy_obce.csv", sep=",", encoding="utf-8-sig")
obce = obce.rename(columns={"KodObce": "KodObec"})

kvartaly = pd.read_csv("data_vystup/rozdeleni_okrsku_kvartaly.csv")
kvartaly["PROCPOC"] = (kvartaly["PROCPOC"] * 100).round(decimals=2)

obce_kvartaly = kvartaly.merge(obce, on="KodObec", how="left")

bins_obce = pd.qcut(obce_kvartaly["PROCPOC"], q=4, retbins=True)[1]
print("Obce:", [round(b, 2) for b in bins_obce])

barvy = {
    1: {'color': '#C1121F', 'opacity': 0.3},
    2: {'color': '#B50D11', 'opacity': 0.5},
    3: {'color': '#C1121F', 'opacity': 0.7},
    4: {'color': '#780000', 'opacity': 1.0},
}
obce_kvartaly['barva'] = obce_kvartaly['KVARTAL'].map(barvy)

# mapa
mapa = folium.Map(location=[49.8, 15.5], zoom_start=8, tiles=None)

# hranice ČR
folium.GeoJson(
    "data_vstup/hranice_cr.geojson",
    style_function=lambda x: {
        "fillColor": "#E9ECEB",
        "color": "#003049",
        "weight": 2,
        "fillOpacity": 1
    }
).add_to(mapa)

# body
for _, radek in obce_kvartaly.iterrows():
    folium.CircleMarker(
        location=[radek["lat"], radek["lon"]],
        radius=3,
        color=barvy[radek["KVARTAL"]]['color'],
        weight=0,
        fill=True,
        fill_color=barvy[radek["KVARTAL"]]['color'],
        fill_opacity=barvy[radek["KVARTAL"]]['opacity'],
        tooltip=f"{radek['NAZEV']}<br>Podíl prefenčních hlasů pro ženy: {radek['PROCPOC']}%"
    ).add_to(mapa)

legenda_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; 
     background-color: #E9ECEB; padding: 15px; border-radius: 8px;
     border: 2px solid #003049; font-family: Arial; color: #003049;">
    <b>Kvartil podílu preferenčních hlasů pro ženy</b><br><br>
    <i style="background:#C1121F; opacity:0.2; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 1 (0,0 – 24,77 %)<br>
    <i style="background:#B50D11; opacity:0.5; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 2 (24,77 – 30,30 %)<br>
    <i style="background:#C1121F; opacity:0.7; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 3 (30,30 – 36,04 %)<br>
    <i style="background:#780000; opacity:1.0; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 4 (36,04 – 100,0 %)<br>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda_html))

mapa.save("mapa_obce.html")
print("Hotovo!")

Obce: [np.float64(0.0), np.float64(24.77), np.float64(30.3), np.float64(36.04), np.float64(100.0)]
Hotovo!


In [26]:
# mapa okrsky

import folium
import pandas as pd

okrsky = pd.read_csv("data_vystup/okrsky_geojson.csv", sep=";") # načíst okrsky - s geodaty

kvartaly = pd.read_csv("data_vystup/rozdeleni_okrsku_kvartaly.csv") # načíst kvartály
kvartaly["PROCPOC"] = (kvartaly["PROCPOC"] * 100).round(decimals=2) 

okrsky_kvartaly = okrsky.merge(kvartaly, on="IdOkrsek", how="left") # spojení oksků s geo a okrsků s kvartály

bins_okrsky = pd.qcut(okrsky_kvartaly["PROCPOC"], q=4, retbins=True)[1]
print("Okrsky:", [round(b, 2) for b in bins_okrsky])

barvy = {
    1: {'color': '#C1121F', 'opacity': 0.2},
    2: {'color': '#B50D11', 'opacity': 0.5},
    3: {'color': '#C1121F', 'opacity': 0.7},
    4: {'color': '#780000', 'opacity': 1.0},
}
okrsky_kvartaly['barva'] = okrsky_kvartaly['KVARTAL'].map(barvy)

# mapa
mapa = folium.Map(location=[49.8, 15.5], zoom_start=8, tiles=None)

# hranice ČR
folium.GeoJson(
    "data_vstup/hranice_cr.geojson",
    style_function=lambda x: {
        "fillColor": "#E9ECEB",
        "color": "#003049",
        "weight": 2,
        "fillOpacity": 1
    }
).add_to(mapa)

# body
for _, radek in okrsky_kvartaly.iterrows():
    nazev_mco = radek["naz_mco"]
    nazev_obec = radek["naz_obec"]
    mco = ("<br>" + nazev_mco) if nazev_mco.strip() != "" else "<br>-"
    procpoc = radek["PROCPOC"]
    tooltip = f"{nazev_obec}{mco}<br>Podíl preferenčních hlasů pro ženy: {procpoc}%"
    folium.CircleMarker(
        location=[radek["lat"], radek["lon"]],
        radius=3,
        color=barvy[radek["KVARTAL"]]['color'],
        weight=0,
        fill=True,
        fill_color=barvy[radek["KVARTAL"]]['color'],
        fill_opacity=barvy[radek["KVARTAL"]]['opacity'],
        tooltip=tooltip
    ).add_to(mapa)

legenda_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; 
     background-color: #E9ECEB; padding: 15px; border-radius: 8px;
     border: 2px solid #003049; font-family: Arial; color: #003049;">
    <b>Kvartil podílu preferenčních hlasů pro ženy</b><br><br>
    <i style="background:#C1121F; opacity:0.2; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 1 (0,0 – 24,77 %)<br>
    <i style="background:#B50D11; opacity:0.5; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 2 (24,77 – 30,30 %)<br>
    <i style="background:#C1121F; opacity:0.7; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 3 (30,30 – 36,04 %)<br>
    <i style="background:#780000; opacity:1.0; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 4 (36,04 – 100,0 %)<br>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda_html))

mapa.save("mapa_okrsky.html")
print("Hotovo!")

Okrsky: [np.float64(0.0), np.float64(24.77), np.float64(30.3), np.float64(36.04), np.float64(100.0)]
Hotovo!


Mapy kvartilů podílů preferenčních hlasů pro ženy v okrscích dle stranických prefencí - liberální blok (Piráti, STAN, SPOLU), konzervativní blok (ANO, SPD, Motoristé)

Liberální blok

In [36]:
# mapa podíly preferenčních hlasů liberální blok (16, 11, 23) - na okrsek
import pandas as pd

df = pd.read_csv("data_vstup/2025_ps_vysledky_okrsky_hlasy_id_pohlavi.csv", low_memory=False)

lib_strany = df[df["KSTRANA"].isin([23, 16, 11])]

# hlasy pro ženy
hlasy_zeny = (
    lib_strany[lib_strany["POHLAVI"] == "ZENA"]
    .groupby(["IdOkrsek", "KSTRANA"])["POC_HLASU"]
    .sum()
    .reset_index()
    .rename(columns={"POC_HLASU": "hlasy_zeny"})
)

# hlasy celkem
hlasy_celkem = (
    lib_strany
    .groupby(["IdOkrsek", "KSTRANA"])["POC_HLASU"]
    .sum()
    .reset_index()
    .rename(columns={"POC_HLASU": "hlasy_celkem"})
)

# součet hlasů přes všechny tři strany dohromady
hlasy_zeny_celkem = hlasy_zeny.groupby("IdOkrsek")["hlasy_zeny"].sum().reset_index()
hlasy_celkem_celkem = hlasy_celkem.groupby("IdOkrsek")["hlasy_celkem"].sum().reset_index()

lib_vysledek = hlasy_zeny_celkem.merge(hlasy_celkem_celkem, on="IdOkrsek")
lib_vysledek["podil_zeny"] = (lib_vysledek["hlasy_zeny"] / lib_vysledek["hlasy_celkem"] * 100).round(2)

# připojit souřadnice
okrsky = pd.read_csv("data_vystup/okrsky_geojson.csv", sep=";")
lib_vysledek = lib_vysledek.merge(okrsky[["IdOkrsek", "lat", "lon", "naz_obec", "naz_mco"]], on="IdOkrsek", how="left")

# rozpočítání výsledků do kvartilů
lib_vysledek["kvartal"] = pd.qcut(lib_vysledek["podil_zeny"], q=4, labels=[1, 2, 3, 4]).astype(int)


In [40]:
# mapa podíly liberální blok

# rozsahy kvartilů
bins = pd.qcut(lib_vysledek["podil_zeny"], q=4, retbins=True)[1]
print([round(b, 2) for b in bins])

import folium

barvy = {
    1: {'color': "#48A6D8", 'opacity': 0.2},
    2: {'color': "#007CBE", 'opacity': 0.5},
    3: {'color': "#005D8F", 'opacity': 0.7},
    4: {'color': "#003049", 'opacity': 1.0},
}

# mapa
mapa = folium.Map(location=[49.8, 15.5], zoom_start=8, tiles=None)

# hranice ČR
folium.GeoJson(
    "data_vstup/hranice_cr.geojson",
    style_function=lambda x: {
        "fillColor": "#E9ECEB",
        "color": "#003049",
        "weight": 2,
        "fillOpacity": 1
    }
).add_to(mapa)

# body
for _, radek in lib_vysledek.iterrows():
    nazev_mco = radek["naz_mco"]
    nazev_obec = radek["naz_obec"]
    mco = ("<br>" + nazev_mco) if nazev_mco.strip() != "" else "<br>-"
    procpoc = radek["podil_zeny"]
    tooltip = f"{nazev_obec}{mco}<br>Podíl preferenčních hlasů pro ženy: {procpoc}%"
    folium.CircleMarker(
        location=[radek["lat"], radek["lon"]],
        radius=3,
        color=barvy[radek["kvartal"]]['color'],
        weight=0,
        fill=True,
        fill_color=barvy[radek["kvartal"]]['color'],
        fill_opacity=barvy[radek["kvartal"]]['opacity'],
        tooltip=tooltip
    ).add_to(mapa)

legenda_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; 
     background-color: #E9ECEB; padding: 15px; border-radius: 8px;
     border: 2px solid #003049; font-family: Arial; color: #003049;">
    <b>Kvartil podílu preferenčních hlasů pro ženy - liberální blok</b><br><br>
    <i style="background:#48A6D8; opacity:0.2; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 1 (15,38 – 26,11 %)<br>
    <i style="background:#007CBE; opacity:0.5; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 2 (26,12 – 29,40 %)<br>
    <i style="background:#005D8F; opacity:0.7; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 3 (29,41 – 32,47 %)<br>
    <i style="background:#003049; opacity:1.0; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 4 (32,48 – 50,0 %)<br>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda_html))

mapa.save("mapa_okrsky_liberal.html")
print("Hotovo!")

[np.float64(15.38), np.float64(26.12), np.float64(29.41), np.float64(32.48), np.float64(50.0)]
Hotovo!


Konzervativní blok

In [38]:
# mapa podíly preferenčních hlasů konzervativní blok (22, 6, 20) - na okrsek
import pandas as pd

df = pd.read_csv("data_vstup/2025_ps_vysledky_okrsky_hlasy_id_pohlavi.csv", low_memory=False)

konz_strany = df[df["KSTRANA"].isin([22, 6, 20])]

# hlasy pro ženy
hlasy_zeny = (
    konz_strany[konz_strany["POHLAVI"] == "ZENA"]
    .groupby(["IdOkrsek", "KSTRANA"])["POC_HLASU"]
    .sum()
    .reset_index()
    .rename(columns={"POC_HLASU": "hlasy_zeny"})
)

# hlasy celkem
hlasy_celkem = (
    konz_strany
    .groupby(["IdOkrsek", "KSTRANA"])["POC_HLASU"]
    .sum()
    .reset_index()
    .rename(columns={"POC_HLASU": "hlasy_celkem"})
)

konz_vysledek = hlasy_zeny.merge(hlasy_celkem, on=["IdOkrsek", "KSTRANA"])
konz_vysledek["podil_zeny"] = (konz_vysledek["hlasy_zeny"] / konz_vysledek["hlasy_celkem"] * 100).round(2)

# součet hlasů přes všechny tři strany dohromady
hlasy_zeny_celkem = hlasy_zeny.groupby("IdOkrsek")["hlasy_zeny"].sum().reset_index()
hlasy_celkem_celkem = hlasy_celkem.groupby("IdOkrsek")["hlasy_celkem"].sum().reset_index()

konz_vysledek = hlasy_zeny_celkem.merge(hlasy_celkem_celkem, on="IdOkrsek")
konz_vysledek["podil_zeny"] = (konz_vysledek["hlasy_zeny"] / konz_vysledek["hlasy_celkem"] * 100).round(2)

# připojit souřadnice
okrsky = pd.read_csv("data_vystup/okrsky_geojson.csv", sep=";")
konz_vysledek = konz_vysledek.merge(okrsky[["IdOkrsek", "lat", "lon", "naz_obec", "naz_mco"]], on="IdOkrsek", how="left")

# rozpočítání výsledků do kvartilů
konz_vysledek["kvartal"] = pd.qcut(konz_vysledek["podil_zeny"], q=4, labels=[1, 2, 3, 4]).astype(int)

# rozsahy kvartilů
bins = pd.qcut(konz_vysledek["podil_zeny"], q=4, retbins=True)[1]
print([round(b, 2) for b in bins])

[np.float64(11.5), np.float64(21.81), np.float64(25.06), np.float64(26.22), np.float64(55.48)]


In [39]:
# mapa podíly konzervativní blok

import folium

barvy = {
    1: {'color': "#BEBEBE", 'opacity': 0.2},
    2: {'color': "#727272", 'opacity': 0.5},
    3: {'color': "#2E2E2E", 'opacity': 0.7},
    4: {'color': "#000000", 'opacity': 1.0},
}


# mapa
mapa = folium.Map(location=[49.8, 15.5], zoom_start=8, tiles=None)

# hranice ČR
folium.GeoJson(
    "data_vstup/hranice_cr.geojson",
    style_function=lambda x: {
        "fillColor": "#E9ECEB",
        "color": "#003049",
        "weight": 2,
        "fillOpacity": 1
    }
).add_to(mapa)

# body
for _, radek in konz_vysledek.iterrows():
    nazev_mco = radek["naz_mco"]
    nazev_obec = radek["naz_obec"]
    mco = ("<br>" + nazev_mco) if nazev_mco.strip() != "" else "<br>-"
    procpoc = radek["podil_zeny"]
    tooltip = f"{nazev_obec}{mco}<br>Podíl preferenčních hlasů pro ženy: {procpoc}%"
    folium.CircleMarker(
        location=[radek["lat"], radek["lon"]],
        radius=3,
        color=barvy[radek["kvartal"]]['color'],
        weight=0,
        fill=True,
        fill_color=barvy[radek["kvartal"]]['color'],
        fill_opacity=barvy[radek["kvartal"]]['opacity'],
        tooltip=tooltip
    ).add_to(mapa)

legenda_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; 
     background-color: #E9ECEB; padding: 15px; border-radius: 8px;
     border: 2px solid #003049; font-family: Arial; color: #003049;">
    <b>Kvartil podílu preferenčních hlasů pro ženy - konzervativní blok</b><br><br>
    <i style="background:#BEBEBE; opacity:0.2; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 1 (11,5 – 21,80 %)<br>
    <i style="background:#727272; opacity:0.5; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 2 (21,81 – 25,04 %)<br>
    <i style="background:#2E2E2E; opacity:0.7; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 3 (25,05 – 26,21 %)<br>
    <i style="background:#000000; opacity:1.0; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 4 (26,22 – 55,48 %)<br>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda_html))

mapa.save("mapa_okrsky_konservativ.html")
print("Hotovo!")

Hotovo!
